# Creating a Custom Pipeline Variant

The library is built around **swappable pipeline variants**. There are three
levels of customization:

1. **Knobs** — configure an existing pipeline with parameters, e.g.
   `Baseline(preprocess=PreprocessConfig(chunk_method="sentence"))`.
2. **Configured subclass** — subclass `Baseline` and only override *defaults*
   in `__init__`. This is exactly how the bundled `Semantic` pipeline is built:
   no stage logic is touched.
3. **Stage override** — subclass and reimplement a stage (`preprocess`,
   `build_kg`, ...) for full control. Keep the build storage-agnostic via
   `build_kg_into` + a `GraphWriter`.

This notebook creates **two variants** — `SmallChunks` (config style) and
`HandRolled` (stage-override style) — registers them in `PIPELINE_REGISTRY`,
benchmarks them against the baseline, and runs one end-to-end.

Everything is **offline-safe**: sentence chunking + minhash dedup + regex /
ontology-rules extraction avoid HuggingFace / spaCy model downloads.

In [ ]:
from pathlib import Path

from polygraph._shared.stage_config import PreprocessConfig
from polygraph.benchmark_pipeline import Benchmark
from polygraph.kg_build import build_kg_into, extract, resolve
from polygraph.kg_build.build import NetworkXGraphWriter
from polygraph.pipelines import PIPELINE_REGISTRY, Baseline
from polygraph.preprocess import chunk, clean, load, quality


def _find_root() -> Path:
    """Repo root = the directory holding pyproject.toml + src/polygraph."""
    for cand in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (cand / "pyproject.toml").exists() and (cand / "src" / "polygraph").is_dir():
            return cand
    raise RuntimeError("Could not find the polygraph repo root above the working directory.")


REPO_ROOT = _find_root()
GOLD = REPO_ROOT / "benchmarks" / "data"

# Offline-safe baseline knobs: sentence chunking + minhash dedup avoid
# HuggingFace model downloads so the tutorial runs anywhere.
BASE = {
    "preprocess": PreprocessConfig(
        chunk_method="sentence",
        doc_dedup_method="minhash",
        chunk_dedup_method="minhash",
    ),
}

In [ ]:
# ── Variant 1: subclass + configure (the "90% case") ───────────
# Subclass Baseline and only override DEFAULTS in __init__. This is exactly
# how the bundled `Semantic` pipeline is built — no stage logic is touched,
# and every other Baseline knob still works.
class SmallChunks(Baseline):
    """Baseline, but with smaller sentence chunks (200 vs 450 tokens)."""

    def __init__(self, **kwargs):
        kwargs.setdefault(
            "preprocess",
            PreprocessConfig(
                chunk_method="sentence",
                chunk_target_tokens=200,
                doc_dedup_method="minhash",
                chunk_dedup_method="minhash",
            ),
        )
        super().__init__(**kwargs)


small = SmallChunks()
print(type(small).__name__, "created — chunk_target_tokens=200")

In [ ]:
# ── Variant 2: override stages directly (full control) ─────────
# Reimplement preprocess() and build_kg() yourself. The build still writes
# through build_kg_into + a GraphWriter, so the graph stays storage-agnostic
# (the same routine Baseline uses for networkx / sqlite / neo4j backends).
class HandRolled(Baseline):
    """A hand-rolled variant: own preprocessing and build logic."""

    def preprocess(self):
        docs = clean.normalize(load.from_paths(self.input_paths))
        chunks = chunk.by_sentence(docs, target_tokens=450, overlap_tokens=60)
        return quality.filter(chunks)

    def build_kg(self, chunks):
        ontology = self._load_ontology()
        entities, triples = extract.with_methods(
            chunks, ontology, entity_method="regex", relation_method="ontology_rules"
        )
        resolved, id_map = resolve.with_method_and_mapping(entities)
        triples = [(id_map.get(s, s), p, id_map.get(o, o), *rest) for (s, p, o, *rest) in triples]
        writer = NetworkXGraphWriter(ontology=ontology)
        build_kg_into(writer, chunks, resolved, triples)
        return {"graph": writer.graph, "entities": resolved, "triples": triples}


hand = HandRolled()
print(type(hand).__name__, "created")

In [ ]:
# ── Register the variants so they are discoverable by name ─────
PIPELINE_REGISTRY["small_chunks"] = SmallChunks
PIPELINE_REGISTRY["hand_rolled"] = HandRolled

print("registered variants:", sorted(PIPELINE_REGISTRY))
# → discoverable by the CLI (`python main.py --variant small_chunks`) and by
#   BenchmarkRunner YAML configs (`variant: small_chunks`).

In [ ]:
# ── Benchmark: variant vs baseline, side by side ───────────────
chunking = Benchmark.Chunking(dataset=GOLD / "chunking_gold.jsonl").run(
    pipelines={
        "baseline": Baseline(**BASE),
        "small_chunks": SmallChunks(),
    },
    max_records=200,
)
print(chunking)

In [ ]:
# ── Run a variant end-to-end on real documents ─────────────────
kg = hand.execute(
    input_paths=[str(REPO_ROOT / "data" / "wikipedia" / "connected.jsonl")],
    output_dir=str(REPO_ROOT / "output" / "tutorial_custom_pipeline"),
)
print(f"KG: {kg['graph'].number_of_nodes()} nodes, {kg['graph'].number_of_edges()} edges")

## Recap & where to go next

- **Three levels of customization:**
  1. *Knobs* — pass configs to a pipeline (`Baseline(preprocess=PreprocessConfig(...))`).
  2. *Configured subclass* — override defaults in `__init__` (like `Semantic` /
     `SmallChunks`).
  3. *Stage override* — reimplement `preprocess` / `build_kg` (like `HandRolled`);
     keep the build storage-agnostic via `build_kg_into` + a `GraphWriter`.
- **Registering** in `PIPELINE_REGISTRY` makes a variant discoverable by the CLI
  (`python main.py --variant <name>`) and by `BenchmarkRunner` YAML configs.
- **Benchmarking** any variant is one call:
  `Benchmark.<Stage>(dataset=...).run(pipelines={...})` — pass as many variants
  as you like, scored side-by-side.

Next steps: [benchmarking.ipynb](benchmarking.ipynb) for all five stage
benchmarks; [kg_storage_agnostic.ipynb](kg_storage_agnostic.ipynb) to build the
same pipeline locally vs Neo4j; [CONTRIBUTING.md](../CONTRIBUTING.md) and
[docs/tutorial.md](../docs/tutorial.md) §8 for the written guide.